In [1]:
from typing import Tuple
import pandas as pd
import glob
import os.path as op
import re
import json
import yaml
import requests

gallica = pd.read_excel("./metadata/monographies-gallica.xlsx")
persee = pd.read_csv("./metadata/metadata-persee.csv")
datacatal = pd.read_csv("./metadata/data-catalogue.csv", names=["ark", "jpg", "Year", "page"])

In [2]:
def get_or_download(ark):
    fpath = f"./.manifests-cache/{ark}.json"
    if not op.exists(fpath):
        try:
            req = requests.get(f"https://gallica.bnf.fr/iiif/ark:/12148/{ark}/manifest.json")
            req.raise_for_status()
        except Exception:
            return None
        with open(fpath, "w") as f:
            f.write(req.text)
    try:
        with open(fpath) as f:
            d = json.load(f)
    except:
        return None
    return d

In [3]:
persee["sub"] = persee["identifier"].apply(lambda x: x.split("/")[-1]) 
persee_ids = persee["sub"].tolist()

In [4]:
datacatal["ark"] = datacatal.ark.apply(lambda x: x.split("/")[-1])
datacatal.jpg = datacatal.jpg.apply(lambda x: x.replace("é", "e"))
dcjpg = datacatal.jpg.tolist()
datacatal

,ark,jpg,Year,page
0,bpt6k1240055q,bienaime-feuardent_12148_bpt6k1240055q_f38.jpg,1866,f38
1,bpt6k1240055q,bienaime-feuardent_12148_bpt6k1240055q_f86.jpg,1866,f86
2,bpt6k1240127r,bienaime-feuardent_12148_bpt6k1240127r_f1.jpg,1899,f1
3,bpt6k1240127r,bienaime-feuardent_12148_bpt6k1240127r_f22.jpg,1899,f22
4,bpt6k12402755,bienaime-feuardent_12148_bpt6k12402755_f13.jpg,1870,f13
...,...,...,...,...
1421,CVP03250_19240411,Lair-Dubreuil_CVP03250_19240411_f6.jpg,1924,f6
1422,CVP03254_19250508,Lair-Dubreuil_CVP03254_19250508_f124.jpg,1925,f124
1423,CVP03254_19250508,Lair-Dubreuil_CVP03254_19250508_f1.jpg,1925,f1
1424,CVP03258_19250612,Lair-Dubreuil_CVP03258_19250612_f177.jpg,1925,f177


In [53]:
re_gallica = re.compile(r"(b[dp]+[A_Za-z0-9]+)_[f]?(\d+)_jp[e]?g")
re_bpt = re.compile(r"((?:bpt6|bd6t|cb)[a-z0-9]+)_")
re_persee = re.compile(r"^[^_]+_[^_]+_(\d{4})_(?:[a-z]{3})_")

data = []
theses = []
downloaded = {}
names = ["MainZone-PLabelled", "Numbering", "MainZone-Head", "MainZone-PStyled", "MainZone-PStructured", "MainZone-Dateline", "MainZone-Item", 
         "RunningTitleZone", "StampZone", "GraphicZone", "GraphicZone-Head", "MarginTextZone-Ab", "MainZone-Address", "MainZone-Continued", "MarginTextZone-PLabelled", 
         "TitlePageZone", "MarginTextZone-ManuscriptAddendum", "DigitizationArtefactZone", "MainZone-Ab", "MainZone-HeadStructured", "GraphicZone-Decoration", "GraphicZone-Ab",
         "GraphicZone-Part", "MarginTextZone-P", "MainZone-Lg", "QuireMarksZone", "MainZone-Signed", "TableZone", "MainZone-PQuoted", "TableZone-Head",
         "GraphicZone-P", "MusicZone", "MainZone-DropCapital", "MarginTextZone-Continued", "MainZone-Maths", "MarginTextZone-Lg", "FigureZone", "FormZone", 
         "FigureZone-Head", "MarginTextZone-PQuoted", "MarginTextZone-PStyled", "FormZone-Head", "MarginTextZone-Item"]

is_gallica = 0

def year_string_to_year_certainty(year: str, default: int = None) -> Tuple[int, bool]:
    certainty = False
    if year is None:
        return year, certainty
    if isinstance(year, int):
        return year, True
        
    if year.isnumeric():
        return int(year), True
        
    if not year.endswith("-") and "-" in year:
        year = year.split("-")[-1]
    
    if year.endswith("-"):
        year = int(year.replace("-", "00"))
    elif year.endswith("."):
        year = int(year.replace(".", "0"))
    elif year.endswith("?"):
        year = int(year.replace("?", "0"))
        
    return year, certainty

for file in glob.glob("../LADAS2-REVIEW/*/*/*.jpg"):
    _, _, split, subset, *_ = file.split("/")
    base, _ = op.splitext(op.basename(file))
    dirname = op.dirname(file)

    defaults = {"file": op.basename(file), "subset": subset, "split": split}
        
    if "theatre" in subset:
        year = re.search(r'_([0-9]{4})_', base)
        data.append({**defaults, "year": year[0], "dating-certainty": False})
        continue
        
    if "monographie" in subset:
        try:
            ark = re_bpt.findall(base)
            entry = gallica[gallica.URL == f"https://gallica.bnf.fr/ark:/12148/{ark[0]}"]
        except IndexError as e:
            year = re.search(r'_([0-9]{4})_', base)
            data.append({**defaults, "year": year[0], "dating-certainty": False})
            continue
        
        if not entry.empty:
            entry = entry.iloc[0].to_dict()
            year, certainty = year_string_to_year_certainty(entry["DATES"])
            data.append({**defaults, "year": year, "dating-certainty": certainty})
            continue
            
    if "others" in subset:
        data.append({**defaults, "year": 1857, "dating-certainty": False})
        continue
        
    if "fingers" in subset:
        data.append({**defaults, "year": 2000, "dating-certainty": False})
        continue
        
    if "these" in subset:
        date = base.replace(("these_"), "")[:4]
        data.append({**defaults, "year": int(date) if date.isnumeric() else None, "dating-certainty": True})
        if data[-1]["year"] is None:
            data[-1]["year"] = 2019 # seems like handle based identifiers are all 2019 
            data[-1]["dating-certainty"] = False # seems like handle based identifiers are all 2019 
        continue
    
    if "typewriter" in subset:
        year = base.replace(("Tapuscrit_"), "")[:4]
        year, certainty = (int(year), True) if year.isnumeric() else (None, False)
        data.append({**defaults, "year": year, "dating-certainty": certainty})
        continue
        
    if "picard" in subset:
        data.append({**defaults, "year": int(base.split("_")[2]), "dating-certainty": True})
        continue
        
    if "magazine-tech" in subset:
        date = base.replace(("magazineJV_"), "")[:4]
        data.append({**defaults, "year": int(date) if date.isnumeric() else None, "dating-certainty": True})
        continue

    if "Le_Prince_Dgem_chronique_dauphinoise" in base:
        defaults["subset"] = "monographies"
        data.append({**defaults, "year": 1860, "dating-certainty": True})
        continue
    
    if "persee" in subset:
        if "dictionnaire" in file:
            century = re.search(r'([0-9]{2})_', file)
            data.append({**defaults, "year":year_string_to_year_certainty(str(year)+'-'), "dating-certainty": True})
            continue
        else:
            year = re_persee.findall(base)[0]
            data.append({**defaults, "year": year_string_to_year_certainty(year)[0], "dating-certainty": True})
            continue

    if "catalogue" in subset:
        year = datacatal[datacatal["jpg"].str.contains(base)]
        if not year.empty:
            year, certainty = year_string_to_year_certainty(year.iloc[0].to_dict()["Year"])
            data.append({**defaults, "year": int(year), "dating-certainty": certainty})   
            continue
        else:
            try:
                ark = re_bpt.findall(base)[0]
                print(file, ark)
            except IndexError as e:
                data.append({**defaults, "year": None, "certainty": False})
                continue
            manifest = get_or_download(ark)
            if not manifest:
                data.append({**defaults, "year": None, "certainty": False})
            else:
                year = ([elem["value"] for elem in manifest["metadata"] if elem["label"] == "Date"] or [None])[0]
                year, certainty = year_string_to_year_certainty(year)
                data.append({**defaults, "year": int(year), "dating-certainty": certainty})
            continue


../LADAS2-REVIEW/test/catalogue-test/bourgey_12148-bpt6k9778130d_f12.jpg bpt6k9778130d
../LADAS2-REVIEW/test/catalogue-test/bienaime-feuardent_12148-bpt6k9778294p_f44.jpg bpt6k9778294p
../LADAS2-REVIEW/test/catalogue-test/rollin_12148-bpt6k9777429g_f3.jpg bpt6k9777429g
../LADAS2-REVIEW/test/catalogue-test/rollin_12148-bpt6k9777328q_f124.jpg bpt6k9777328q
../LADAS2-REVIEW/test/catalogue-test/rollin_12148-bpt6k97774215_f74.jpg bpt6k97774215
../LADAS2-REVIEW/test/catalogue-test/bourgey_12148-bpt6k97773687_f28.jpg bpt6k97773687
../LADAS2-REVIEW/test/catalogue-test/bienaime-feuardent_12148-bpt6k9778183j_f65.jpg bpt6k9778183j
../LADAS2-REVIEW/test/catalogue-test/bourgey_12148-bpt6k9781827f_f2.jpg bpt6k9781827f
../LADAS2-REVIEW/test/catalogue-test/rollin_12148-bpt6k97791900_f20.jpg bpt6k97791900
../LADAS2-REVIEW/test/catalogue-test/bienaime-feuardent_12148-bpt6k9778560x_f79.jpg bpt6k9778560x
../LADAS2-REVIEW/test/catalogue-test/bienaime-feuardent_12148-bpt6k97793750_f67.jpg bpt6k97793750
../L

TypeError: 'NoneType' object is not subscriptable

In [50]:
pd.DataFrame(data).to_csv("metadata.csv", index=False)

In [13]:
print(pd.DataFrame(data).pivot_table(index="subset", values="year", columns="split", aggfunc="count").fillna(0)[["train", "valid", "test"]].to_markdown())

KeyError: "None of [Index(['train', 'valid', 'test'], dtype='object', name='split')] are in the [columns]"

In [ ]:

    if "Picard_Concours_" in base:
        print(base)
        date = base.replace(("Picard_Concours_"), "")[:4]
        data.append({
            "file": op.basename(file),
            "year": int(date) if date.isnumeric() else None,
            "dating-certainty": True,
            "subset": "Agence Picard"
        })
    elif "sohim_0398-3811_2007_edc_34_1_907_0457" in base:
        data.append({
            "file": op.basename(file),
            "year": 2007,
            "dating-certainty": True,
            "subset": "Persée"
        })
    else:
        if pid in persee_ids:
            data.append({
                "file": op.basename(file),
                "year": persee[persee["sub"] == base].iloc[0].to_dict()["date"],
                "dating-certainty": True,
                "subset": "Persée"
            })
            continue
        if "catalogue" in file:
            year = datacatal[datacatal["jpg"] == op.basename(file)]
            if not year.empty:
                year = year.iloc[0].to_dict()["Year"]
                certainty = True
                if "-" in year:
                    year = year.split("-")[-1]
                    certainty = False
                if year.endswith(".."):
                    year = year.replace(".", "0")
                    certainty = False
                data.append({
                    "file": op.basename(file),
                    "dating": int(year),
                    "certainty": certainty,
                    "subset": "DataCatalogue"
                })   
                continue
        else:
            if group := re_bpt.findall(base):
                group = group[0]
                year = datacatal[datacatal["ark"] == group].head(1)["Year"]
                if not year.empty:
                    year = year.tolist()[0]
                    data.append({
                        "file": op.basename(file),
                        "year": datacatal[datacatal["ark"] == group].head(1)["Year"].iloc[0],
                        "dating-certainty": True,
                        "subset": "DataCatalogue"
                    })
                else:
                    certainty = True
                    j = get_or_download(group)
                    if not j:
                        data.append({
                            "file": op.basename(file),
                            "year": None,
                            "dating-certainty": certainty,
                            "subset": "Monographies"
                        })
                    else:
                        for elem in j["metadata"]:
                            if elem["label"] == "Date":
                                year = elem["value"]
                                if "-" in year:
                                    year = year.split("-")[-1]
                                    certainty = False
                                if "." in year:
                                    year = year.replace(".", "0")
                                    certainty = False
                                if "?" in year:
                                    year = year.replace("?", "0")
                                    certainty = False
                                year = int(year)
                                break
                        data.append({
                            "file": op.basename(file),
                            "year": year,
                            "dating-certainty": certainty,
                            "subset": "Monographies"
                        })
                continue # print(base)
            else:
                print(file)